# Create a Content Understanding Analyzer

Use this notebook to create the Lab 04 Content Understanding analyzer step by step. The notebook follows `create-analyzer.py`, but breaks the flow into smaller cells so you can inspect configuration, review the analyzer schema, and then create or replace the analyzer when ready.

## 1. Import Required Packages

The analyzer is created with the Azure AI Content Understanding SDK. Configuration is loaded from the shared `workshop/.env` file.

In [1]:
import json
import os
from pathlib import Path

from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

## 2. Locate the Workshop Environment File

This cell finds `workshop/.env` whether the notebook is started from the repository root or from the Lab 04 folder.

In [2]:
def find_workshop_dir(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        workshop_dir = candidate / "workshop"
        if (workshop_dir / ".env").exists():
            return workshop_dir
        if candidate.name == "workshop" and (candidate / ".env").exists():
            return candidate
    raise FileNotFoundError("Could not find workshop/.env. Create it from workshop/.env.sample first.")

workshop_dir = find_workshop_dir()
env_path = workshop_dir / ".env"
lab_dir = workshop_dir / "lab-04-rag-pipeline"

print(f"Workshop directory: {workshop_dir}")
print(f"Environment file: {env_path}")
print(f"Lab directory: {lab_dir}")

Workshop directory: C:\Users\algut\repos\alesaez\content-processing-solution\workshop
Environment file: C:\Users\algut\repos\alesaez\content-processing-solution\workshop\.env
Lab directory: C:\Users\algut\repos\alesaez\content-processing-solution\workshop\lab-04-rag-pipeline


## 3. Load and Validate Configuration

The analyzer creation step requires the Foundry endpoint and key from the parent Foundry resource. The key is not printed.

In [3]:
def required_setting(name: str) -> str:
    value = (os.getenv(name) or "").strip()
    if not value or value.startswith("<"):
        raise ValueError(f"Set {name} in workshop/.env")
    return value

load_dotenv(env_path, override=True)

foundry_endpoint = required_setting("FOUNDRY_ENDPOINT")
foundry_key = required_setting("FOUNDRY_KEY")
analyzer_id = "rag_document_analyzer"

print(f"Foundry endpoint: {foundry_endpoint}")
print(f"Analyzer ID: {analyzer_id}")

Foundry endpoint: https://4ublv6yuwkfh4-aifoundry.services.ai.azure.com/
Analyzer ID: rag_document_analyzer


## 4. Define the Analyzer Schema

The analyzer uses `prebuilt-document` as the base analyzer and asks Content Understanding to generate a short summary and key topics for each document.

In [4]:
analyzer_definition = {
    "description": "Analyzer for extracting structured content from travel documents",
    "baseAnalyzerId": "prebuilt-document",
    "models": {
        "completion": "gpt-4.1",
        "embedding": "text-embedding-3-large",
    },
    "config": {
        "returnDetails": True,
    },
    "fieldSchema": {
        "fields": {
            "Summary": {
                "type": "string",
                "method": "generate",
                "description": "A brief summary of the document content",
            },
            "KeyTopics": {
                "type": "array",
                "method": "generate",
                "description": "Key topics or themes covered in the document",
                "items": {
                    "type": "string",
                },
            },
        },
    },
}

print(json.dumps(analyzer_definition, indent=2))

{
  "description": "Analyzer for extracting structured content from travel documents",
  "baseAnalyzerId": "prebuilt-document",
  "models": {
    "completion": "gpt-4.1",
    "embedding": "text-embedding-3-large"
  },
  "config": {
    "returnDetails": true
  },
  "fieldSchema": {
    "fields": {
      "Summary": {
        "type": "string",
        "method": "generate",
        "description": "A brief summary of the document content"
      },
      "KeyTopics": {
        "type": "array",
        "method": "generate",
        "description": "Key topics or themes covered in the document",
        "items": {
          "type": "string"
        }
      }
    }
  }
}


## 5. Create the Content Understanding Client

This cell creates the SDK client. It does not create the analyzer yet.

In [5]:
client = ContentUnderstandingClient(
    endpoint=foundry_endpoint,
    credential=AzureKeyCredential(foundry_key),
)

print("Content Understanding client is ready.")

Content Understanding client is ready.


## 6. Create or Replace the Analyzer

Run this cell when you are ready to create the analyzer in Azure. The `allow_replace=True` setting makes the cell repeatable while you are working through the lab.

In [6]:
print(f"Creating analyzer '{analyzer_id}'...")

poller = client.begin_create_analyzer(
    analyzer_id=analyzer_id,
    resource=analyzer_definition,
    allow_replace=True,
)

result = poller.result()
status = result.get("status", "Succeeded") if isinstance(result, dict) else "Succeeded"

print(f"Analyzer '{analyzer_id}' created successfully.")
print(f"Status: {status}")

Creating analyzer 'rag_document_analyzer'...
Analyzer 'rag_document_analyzer' created successfully.
Status: Succeeded


## 7. Next Step

After the analyzer is created, return to the Lab 04 README and run the ingestion pipeline:

```powershell
python workshop/lab-04-rag-pipeline/ingest-pipeline.py
```